In [1]:
import urllib.request

print("Downloading pose model...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
    "pose_landmarker.task"
)

print("Downloading hand model...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
    "hand_landmarker.task"
)

print("Done! Both models downloaded.")

Done! Both models downloaded.


In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions
from mediapipe.tasks.python.vision import (
    PoseLandmarker, PoseLandmarkerOptions,
    HandLandmarker, HandLandmarkerOptions,
    RunningMode
)

# ── Setup Pose ──────────────────────────────────────────────
pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker.task'),
    running_mode=RunningMode.VIDEO
)

# ── Setup Hands ─────────────────────────────────────────────
hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=RunningMode.VIDEO,
    num_hands=2
)

cap = cv2.VideoCapture(0)

with PoseLandmarker.create_from_options(pose_options) as pose, \
     HandLandmarker.create_from_options(hand_options) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))

        # Convert to MediaPipe Image
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                            data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # ── Run detection ──
        pose_result = pose.detect_for_video(mp_image, timestamp_ms)
        hand_result = hands.detect_for_video(mp_image, timestamp_ms)

        # ── Draw Pose landmarks ──
        if pose_result.pose_landmarks:
            for landmarks in pose_result.pose_landmarks:
                for lm in landmarks:
                    h, w = frame.shape[:2]
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

        # ── Draw Hand landmarks ──
        if hand_result.hand_landmarks:
            CONNECTIONS = mp.solutions.hands.HAND_CONNECTIONS  # still available for connections only
            for landmarks in hand_result.hand_landmarks:
                for lm in landmarks:
                    h, w = frame.shape[:2]
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)

        cv2.imshow("MediaPipe - New API", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

AttributeError: module 'mediapipe' has no attribute 'solutions'